# Capitolo 16 – Master theorem: chiamate e ricorrenze a confronto con la soluzione (Python)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap16_master_theorem_python.ipynb)

Il capitolo non contiene algoritmi con pseudocodice: questo notebook è quindi breve e **misura** quello che il Master theorem predice. Prima si contano le chiamate di `sumRecHalving`, l'algoritmo di somma per dimezzamento del codice del corso, la cui ricorrenza $T(n)=2T(n/2)+1$ cade nel caso 1 ($\Theta(n)$, in realtà esattamente $2n-1$ chiamate); poi si calcolano direttamente alcune ricorrenze $T(n)=aT(n/b)+f(n)$ degli esempi del capitolo e si verifica che il rapporto con la funzione data dal Master theorem tende a una costante.

Il capitolo completo, con pseudocodice, dimostrazioni ed esempi, è [sul sito](https://fabiofurini.github.io/algoritmi/analisi-ricorsione/16-master-theorem/).

Ogni cella di codice si esegue con **Maiusc+Invio**; le celle vanno eseguite in ordine, dall'alto in basso. Python 3, nessuna libreria esterna.

**Indici.** Nello pseudocodice delle dispense gli array sono indicizzati da 1; nel codice (come in tutti i linguaggi qui usati) da 0: l'elemento `a[i]` dello pseudocodice è l'elemento di indice `i-1` nel codice, e un intervallo `a[l..r]` diventa `a[l-1..r-1]`. Le stampe dei notebook usano gli indici da 0.

**Nomi.** Le funzioni hanno il nome dello pseudocodice, nella convenzione del linguaggio (come nei capitoli di introduzione): `sumRecHalving` → `sum_rec_halving` (snake_case).

## 1. Le chiamate di `sumRecHalving` (caso 1)

`sumRecHalving`$(a,l,r)$ — *Input*: un array e due indici $l\le r$. *Output*: la somma di `a[l..r]`, ottenuta sommando le somme delle due metà. Ogni chiamata fa un lavoro costante, quindi $T(n)=2T(n/2)+1$: $a=2$, $b=2$, $n^{\log_b a}=n$, caso 1, $T(n)=\Theta(n)$. Il capitolo riporta $15$, $1\,999$ e $2\,097\,151$ chiamate per $n=8$, $1000$ e $2^{20}$, cioè esattamente $2n-1$.

Il vettore contiene $1,2,\dots,n$, quindi la somma attesa è $n(n+1)/2$.

In [1]:
calls = 0   # numero di chiamate

def sum_rec_halving(a, l, r):
    """Somma di a[l..r] (indici da 0) per dimezzamento."""
    global calls
    calls += 1
    if l == r:
        return a[l]
    mid = l + (r - l) // 2
    return sum_rec_halving(a, l, mid) + sum_rec_halving(a, mid + 1, r)

for n in [8, 1000, 2 ** 20]:
    a = list(range(1, n + 1))
    calls = 0
    s = sum_rec_halving(a, 0, n - 1)
    print(f"n={n:<8d} somma={s:<14d} chiamate={calls:<9d} 2n-1={2 * n - 1}")

n=8        somma=36             chiamate=15        2n-1=15
n=1000     somma=500500         chiamate=1999      2n-1=1999
n=1048576  somma=549756338176   chiamate=2097151   2n-1=2097151


**Prova tu.** Cambia `n` (anche un valore che non è una potenza di 2) e verifica che le chiamate siano sempre $2n-1$.

In [2]:
# Prova con un n che non è una potenza di 2: le chiamate sono ancora 2n-1.
n = 777
a = list(range(1, n + 1))
calls = 0
print(sum_rec_halving(a, 0, n - 1), calls, 2 * n - 1)

302253 1553 1553


## 2. Ricorrenze degli esempi del capitolo e Master theorem

Per $T(n)=a\,T(n/b)+f(n)$, $T(1)=1$ e $n=b^k$, la funzione `valuta` calcola $T(n)$ dalla definizione ($k$ passi: $T(b^{i})=a\,T(b^{i-1})+f(b^{i})$). Il Master theorem dà l'ordine di grandezza $\Theta(g)$ con $f(n)=n^p$ e $n^{\log_b a}=a^k$:

- **caso 1** ($f$ cresce meno): $g=n^{\log_b a}=a^k$;
- **caso 2** ($f$ e $n^{\log_b a}$ dello stesso ordine): $g=f(n)\cdot\log_b n=n^p\,k$;
- **caso 3** ($f$ cresce di più): $g=f(n)=n^p$.

L'ultima colonna è $T(n)/g(n)$: se il Master theorem è corretto, è un numero limitato e lontano da 0 (per $n$ grande si avvicina a una costante).

In [3]:
def valuta(a, b, p, k):
    """T(n) = a*T(n/b) + n^p, T(1) = 1, per n = b^k."""
    n, t = 1, 1
    for _ in range(k):
        n *= b
        t = a * t + n ** p
    return n, t

# (ricorrenza, a, b, p, k, caso, tipo di g, nome di g)
righe = [
    ("T(n)=T(n/2)+1",      1, 2, 0, 20, 2, "level",  "log n"),
    ("T(n)=2T(n/2)+1",     2, 2, 0, 20, 1, "leaves", "n"),
    ("T(n)=2T(n/2)+n",     2, 2, 1, 20, 2, "level",  "n log n"),
    ("T(n)=3T(n/2)+n",     3, 2, 1, 20, 1, "leaves", "n^log2(3)"),
    ("T(n)=7T(n/2)+n^2",   7, 2, 2, 20, 1, "leaves", "n^log2(7)"),
    ("T(n)=16T(n/4)+n",   16, 4, 1, 10, 1, "leaves", "n^2"),
    ("T(n)=8T(n/2)+n^3",   8, 2, 3, 15, 2, "level",  "n^3 log n"),
    ("T(n)=8T(n/3)+n^2",   8, 3, 2, 12, 3, "f",      "n^2"),
]
print(f"{'ricorrenza':<18s} caso {'n':>8s} {'T(n)':>20s}  {'Theta(g)':<10s} T/g")
for nome, a, b, p, k, caso, tipo, gname in righe:
    n, t = valuta(a, b, p, k)
    g = a ** k if tipo == "leaves" else (n ** p * k if tipo == "level" else n ** p)
    print(f"{nome:<18s} {caso:>4d} {n:>8d} {t:>20d}  {gname:<10s} {t / g:.3f}")

ricorrenza         caso        n                 T(n)  Theta(g)   T/g
T(n)=T(n/2)+1         2  1048576                   21  log n      1.050
T(n)=2T(n/2)+1        1  1048576              2097151  n          2.000
T(n)=2T(n/2)+n        2  1048576             22020096  n log n    1.050
T(n)=3T(n/2)+n        1  1048576          10458256051  n^log2(3)  2.999
T(n)=7T(n/2)+n^2      1  1048576   186180488678924301  n^log2(7)  2.333
T(n)=16T(n/4)+n       1  1048576        1466015154176  n^2        1.333
T(n)=8T(n/2)+n^3      2    32768      562949953421312  n^3 log n  1.067
T(n)=8T(n/3)+n^2      3   531441        1992110014441  n^2        7.053


**Prova tu.** Cambia i parametri $a$, $b$, $p$, $k$ e confronta $T(n)$ con la funzione del caso del Master theorem.

In [4]:
# Prova una ricorrenza tua: T(n) = 4T(n/2) + n (caso 1, g = n^2) con k = 10.
n, t = valuta(4, 2, 1, 10)
print(n, t, f"{t / 4 ** 10:.3f}")

1024 2096128 1.999
